In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from scipy.stats import uniform, loguniform, norm
from sklearn.gaussian_process.kernels import Matern, RBF, WhiteKernel, ConstantKernel as C
from sklearn.preprocessing import power_transform
from scipy.optimize import minimize
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score

# **Function 5** - Chemical Process Optimisation
- A black box function which represents the yield of a chemical process in a factory. The function is typically unimodal, with a single peak where yield is maximised.

- Main goal - find the optimal combination of chemical inputs that delivers the highest possible yield, using systematic exploration and optimisation methods.

- **Goal** - Maximise

In [2]:
# Load current cumulative dataset (original + prior weekly updates) for Function 5
X = np.load('/Users/prathamyeole/Library/CloudStorage/OneDrive-Personal/Imperial - Machine Learning and Artifical Intelligence Certification/Capstone/Weekly Capstone Updates and Comments/Week 1/Initial_data_points_starter/initial_data/function_5/initial_inputs.npy')
Y = np.load('/Users/prathamyeole/Library/CloudStorage/OneDrive-Personal/Imperial - Machine Learning and Artifical Intelligence Certification/Capstone/Weekly Capstone Updates and Comments/Week 1/Initial_data_points_starter/initial_data/function_5/initial_outputs.npy')

# New data for Week 7 (Function 5)
X_w6_new_point = np.array([0.114475, 0.992539, 0.999989, 0.525254], dtype=np.float64)
Y_w6_new_point = np.array([1728.1697427584952], dtype=np.float64)

# Append the new data point
X_updated = np.vstack((X, X_w6_new_point.reshape(1, -1)))

# Remove duplicate X rows
X_unique, unique_indices = np.unique(X_updated, axis=0, return_index=True)

# Build Y and keep matching rows only
Y_all = np.append(Y, Y_w6_new_point)
Y_updated = Y_all[unique_indices]

# Save updated arrays
np.save('/Users/prathamyeole/Library/CloudStorage/OneDrive-Personal/Imperial - Machine Learning and Artifical Intelligence Certification/Capstone/Weekly Capstone Updates and Comments/Week 1/Initial_data_points_starter/initial_data/function_5/initial_inputs.npy', X_unique)
np.save('/Users/prathamyeole/Library/CloudStorage/OneDrive-Personal/Imperial - Machine Learning and Artifical Intelligence Certification/Capstone/Weekly Capstone Updates and Comments/Week 1/Initial_data_points_starter/initial_data/function_5/initial_outputs.npy', Y_updated)

# Print arrays
print("Updated Inputs (X) - Function 5: ", X_unique)
print("Updated Outputs (Y) - Function 5: ", Y_updated)

Updated Inputs (X) - Function 5:  [[0.004861   0.982935   0.98162    0.958237  ]
 [0.028538   0.096927   0.128764   0.401407  ]
 [0.114475   0.992539   0.999989   0.525254  ]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.19144708 0.03819337 0.60741781 0.41458414]
 [0.221849   0.896815   0.894595   0.898055  ]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.236596   0.865457   0.89289    0.897109  ]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.55421    0.123984   0.678221   0.890122  ]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.67749115 0.35850951 0.

## **Output Analysis**

- Last week (W5) I saw 3475.88, and this week (W6) I saw 1728.17. This is a significant drop from the all-time best.

- **Real-World Implication**: Despite keeping x2 and x3 very high (0.993 and 0.999), the W6 yield dropped nearly in half compared to W5. 
    - In a chemical process context this tells us that the yield is highly sensitive to the exact combination of all four inputs simultaneously, changing x1 from 0.005 to 0.114 and x4 from 0.958 to 0.525 was enough to move us away from the peak despite the other inputs being similar.

- The pattern is now very clear. 
    - The three best results (w3, w5, and w6) all share low x1 (near zero) and high x2, x3 (near 1). 
        - The critical difference between W5 (best) and W6 (lower) is x4: W5 had x4=0.958, W6 had x4=0.525. 
            - This strongly suggests x4 needs to be high, close to 0.95+, to achieve maximum yield. 

- The W7 strategy must return as close as possible to the W5 coordinate profile: x1 very low, x2 and x3 very high, and x4 back up near 0.95.

## **Bayesian Optimisation**

- I continue to use an ARD Matern kernel with nu=2.5. 
    - ARD is essential here because the history has now confirmed that the four chemical inputs have very different sensitivities, x1 needs to be near zero, x2 and x3 near 1, and x4 near 0.95.
    - ARD allows the model to learn these independent length scales for each dimension.

- This week I am also introducing the Output Power Transformation from HEBO (1st place, NeurIPS 2020 BBO Challenge). Our outputs now span from 142.9 to 3475.9, a range where the W5 peak is roughly 24 times larger than the W1 and W4 baseline results. 

    - With normalize_y=True, the GP's internal scaling is dominated by this spike, which compresses the meaningful gradient information between W2, W3, W5 and W6. 
    
        - The Yeo-Johnson transform handles this by proportionally scaling the output range, allowing the GP to correctly distinguish the gradient between all the high-yield results rather than just seeing one enormous spike and everything else as flat.

- The ARD kernel and EI acquisition stay the same, they correctly identified the W5 peak and remain the right tool for a unimodal function in exploitation mode. 
    - The power transformation is a calibration upgrade that makes the GP's gradient estimates more reliable given the wide output range.

- To allow the GP to use the full history, particularly the contrast between W5 (x4=0.958, yield=3475) and W6 (x4=0.525, yield=1728), to steer W7 back toward the W5 coordinate profile, especially recovering x4 toward the 0.95+ range.

In [4]:
# Output Power Transformation (HEBO)
Y_transformed = power_transform(
    Y_updated.reshape(-1, 1),
    method='yeo-johnson'
).ravel()

print("Original Y:", Y_updated)
print("Transformed Y:", Y_transformed)

# Define the model
kernel = C(1.0) * Matern(length_scale=[0.1, 0.1, 0.1, 0.1], nu=2.5) + WhiteKernel(noise_level=1e-5)

model = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=25,
    normalize_y=False 
)

model.fit(X_unique, Y_transformed)

Original Y: [3.47588113e+03 1.42925081e+02 1.72816974e+03 4.31612757e+02
 9.97233189e+00 8.84799176e+00 6.44434399e+01 1.52269614e+03
 1.08885962e+03 1.33955718e+03 6.34767158e+01 1.09571876e+02
 4.51815703e+01 1.12939795e-01 3.55806818e+02 2.33223610e+02
 7.97291299e+01 5.75715369e+01 1.55727657e+02 6.44201468e+01
 2.44230883e+01 7.84343889e+01 4.21089813e+00 2.88667516e+01
 1.83013796e+01 2.58370525e+02]
Transformed Y: [ 1.94933385  0.15829243  1.5413682   0.75755143 -1.16664232 -1.22003353
 -0.25856576  1.46841387  1.27652996  1.39485286 -0.26634082  0.01772632
 -0.43976266 -2.25981076  0.65118535  0.42096319 -0.14851866 -0.31643892
  0.20397391 -0.25875179 -0.74556113 -0.15702002 -1.53081711 -0.66358843
 -0.8848088   0.47646934]


/Users/prathamyeole/Library/Python/3.13/lib/python/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


,"kernel kernel: kernel instance, default=NoneThe kernel specifying the covariance function of the GP. If None ispassed, the kernel ``ConstantKernel(1.0, constant_value_bounds=""fixed"")* RBF(1.0, length_scale_bounds=""fixed"")`` is used as default. Note thatthe kernel hyperparameters are optimized during fitting unless thebounds are marked as ""fixed"".",1**2 * Matern...e_level=1e-05)
,"alpha alpha: float or ndarray of shape (n_samples,), default=1e-10Value added to the diagonal of the kernel matrix during fitting.This can prevent a potential numerical issue during fitting, byensuring that the calculated values form a positive definite matrix.It can also be interpreted as the variance of additional Gaussianmeasurement noise on the training observations. Note that this isdifferent from using a `WhiteKernel`. If an array is passed, it musthave the same number of entries as the data used for fitting and isused as datapoint-dependent noise level. Allowing to specify thenoise level directly as a parameter is mainly for convenience andfor consistency with :class:`~sklearn.linear_model.Ridge`.For an example illustrating how the alpha parameter controlsthe noise variance in Gaussian Process Regression, see:ref:`sphx_glr_auto_examples_gaussian_process_plot_gpr_noisy_targets.py`.",1e-10
,"optimizer optimizer: ""fmin_l_bfgs_b"", callable or None, default=""fmin_l_bfgs_b""Can either be one of the internally supported optimizers for optimizingthe kernel's parameters, specified by a string, or an externallydefined optimizer passed as a callable. If a callable is passed, itmust have the signature:: def optimizer(obj_func, initial_theta, bounds): # * 'obj_func': the objective function to be minimized, which # takes the hyperparameters theta as a parameter and an # optional flag eval_gradient, which determines if the # gradient is returned additionally to the function value # * 'initial_theta': the initial value for theta, which can be # used by local optimizers # * 'bounds': the bounds on the values of theta .... # Returned are the best found hyperparameters theta and # the corresponding value of the target function. return theta_opt, func_minPer default, the L-BFGS-B algorithm from `scipy.optimize.minimize`is used. If None is passed, the kernel's parameters are kept fixed.Available internal optimizers are: `{'fmin_l_bfgs_b'}`.",'fmin_l_bfgs_b'
,"n_restarts_optimizer n_restarts_optimizer: int, default=0The number of restarts of the optimizer for finding the kernel'sparameters which maximize the log-marginal likelihood. The first runof the optimizer is performed from the kernel's initial parameters,the remaining ones (if any) from thetas sampled log-uniform randomlyfrom the space of allowed theta-values. If greater than 0, all boundsmust be finite. Note that `n_restarts_optimizer == 0` implies that onerun is performed.",25
,"normalize_y normalize_y: bool, default=FalseWhether or not to normalize the target values `y` by removing the meanand scaling to unit-variance. This is recommended for cases wherezero-mean, unit-variance priors are used. Note that, in thisimplementation, the normalisation is reversed before the GP predictionsare reported... versionchanged:: 0.23",False
,"copy_X_train copy_X_train: bool, default=TrueIf True, a persistent copy of the training data is stored in theobject. Otherwise, just a reference to the training data is stored,which might cause predictions to change if the data is modifiedexternally.",True
,"n_targets n_targets: int, default=NoneThe number of dimensions of the target values. Used to decide the numberof outputs when sampling from the prior distributions (i.e. calling:meth:`sample_y` before :meth:`fit`). This parameter is ignored once:meth:`fit` has been called... versionadded:: 1.3",None
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation used to initialize the centers.Pass an int for reproducible results across multiple function calls.See :term:`

## **Acquisition Function**

- I am using Expected Improvement (EI) with $\xi=0.001$, unchanged from W6.

- I am maintaining $\xi=0.001$. This is a deliberate decision based on the history.

    - The w5 result at 3475.88 remains the undisputed all-time best. 
        - The function is unimodal, meaning there is one peak and we have already been on it.

    - The W6 drop tells us we moved away from the peak by changing x1 and x4.
        - The correct response is not to explore further, it is to return precisely to the W5 profile.

    - A very low $\xi=0.001$ keeps EI tightly focused on finding improvements over w5, which means it will naturally gravitate back toward the W5 coordinate neighbourhood rather than jumping away.

- To return the W7 query as close as possible to the W5 sweet spot, particularly recovering x4 toward 0.95+ while keeping x1 near zero and x2, x3 near 1.

In [6]:
def expected_improvement(X, model, y_max, xi=0.001):
    mu, sigma = model.predict(X, return_std=True)
    mu, sigma = mu.reshape(-1, 1), sigma.reshape(-1, 1)

    with np.errstate(divide='ignore'):
        improvement = mu - y_max - xi
        Z = improvement / sigma
        ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
        ei[sigma <= 0] = 0.0
    return ei.ravel()

# Current best result remains W5 (3475.88)
y_max = np.max(Y_transformed)

x_grid = np.random.uniform(0, 1, (10000, 4))

# Calculate EI values
ei_values = expected_improvement(x_grid, model, y_max)

# Select next query
next_query = x_grid[np.argmax(ei_values)]

print(f"Strategic Week 7 Query (Function 5): [{next_query[0]:.6f}-{next_query[1]:.6f}-{next_query[2]:.6f}-{next_query[3]:.6f}]")

Strategic Week 7 Query (Function 5): [0.841103-0.989215-0.929196-0.999673]
